# Validate epoch evaluation and checkpointing on TPU v5e-1

[Open in Colab](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/main/notebooks/TrainLM_TPU_Epoch_Eval_Save_V5e1.ipynb)

Run this from Colab connected to your one-device v5e-1 Cloud TPU VM. The notebook does not install or replace packages. It detects the available backend and prefers native TorchTPU (`torch-tpu`); it can also run the legacy PyTorch/XLA path (`torch-xla`).

It runs focused CPU contract tests, then combined epoch evaluation/checkpointing, evaluation-only, checkpoint-only, and resume from the first epoch checkpoint. The local tiny model and packed data avoid Hub downloads. The report says which backend actually ran: only the `pytorch_xla` mode exercises PR #143's legacy coordinator; native TorchTPU is a separate path. Hardware generation is recorded as owner-provided information, not runtime-verified.


## 1. Get TrainLM source

Run this after the notebook update has merged to `main`. The printed revision is included in the evidence report.


In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

repo_root = Path(tempfile.mkdtemp(prefix="trainlm-xla-epoch-")) / "TrainLM"
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", "main",
     "https://github.com/Dhiraj309/TrainLM.git", str(repo_root)],
    check=True,
)
source_revision = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
).strip()
sys.path.insert(0, str(repo_root / "src"))
print(f"source_revision={source_revision}")


## 2. Run CPU contract tests

These tests check public argument forwarding, worker flags, and epoch-boundary trainer behavior without initializing a TPU. They are separate from the hardware run below.


In [ ]:
import importlib.util
import os
import subprocess

if importlib.util.find_spec("pytest") is None:
    raise RuntimeError("pytest is unavailable; this notebook does not install packages.")

test_paths = [
    "tests/test_api_tpu_coordinator.py",
    "tests/test_tpu_entrypoint.py",
    "tests/test_api_lifecycle.py",
    "tests/test_finetuning_api.py",
    "tests/contracts/test_roadmap_status.py",
    "tests/training/test_trainer.py",
]
env = os.environ.copy()
env["PYTHONPATH"] = str(repo_root / "src") + os.pathsep + env.get("PYTHONPATH", "")
cpu_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--tb=short", *test_paths],
    cwd=repo_root, env=env, text=True, capture_output=True, timeout=1200,
)
print(cpu_tests.stdout[-6000:])
if cpu_tests.returncode:
    print(cpu_tests.stderr[-3000:])
    raise RuntimeError(f"CPU contract tests failed with exit code {cpu_tests.returncode}.")
print(f"cpu_contract_tests=PASS source_revision={source_revision}")


## 3. Run the TPU checks

Confirm that the connected Cloud TPU VM is `ct5lp-hightpu-1t`. `BACKEND="auto"` selects native TorchTPU when available, matching the TorchTPU Colab runtime; if only PyTorch/XLA is installed, it selects the XLA coordinator. Set `BACKEND` explicitly if you need one path. No backend package is installed by this notebook.


In [ ]:
import importlib.metadata
import importlib.util
import json
import tempfile
from pathlib import Path

import torch
from transformers import GPT2Config, GPT2LMHeadModel
from trainlm import PackedBinDataset, TrainLMTrainer, TrainLMTrainingArguments
from trainlm.data.writer import write_packed_binary_shard
from trainlm.training import TrainerCallback

# Confirm this in the Cloud TPU VM page before leaving the value unchanged.
TPU_VM_MACHINE_TYPE = "ct5lp-hightpu-1t"
BACKEND = "auto"  # "auto", "native_tpu", or "tpu" (legacy PyTorch/XLA)
if TPU_VM_MACHINE_TYPE != "ct5lp-hightpu-1t":
    raise RuntimeError("This validation notebook is scoped to a v5e-1 Cloud TPU VM.")
has_torch_tpu = importlib.util.find_spec("torch_tpu") is not None
has_torch_xla = importlib.util.find_spec("torch_xla") is not None
if BACKEND == "auto":
    if has_torch_tpu:
        BACKEND = "native_tpu"
    elif has_torch_xla:
        BACKEND = "tpu"
    else:
        raise RuntimeError("No TorchTPU or PyTorch/XLA runtime is installed.")
if BACKEND == "native_tpu" and not has_torch_tpu:
    raise RuntimeError("BACKEND='native_tpu' requires the runtime-provided torch-tpu package.")
if BACKEND == "tpu" and not has_torch_xla:
    raise RuntimeError("BACKEND='tpu' requires runtime-provided PyTorch/XLA.")
print(f"selected_backend={BACKEND}")

validation_root = Path(tempfile.mkdtemp(prefix="trainlm-epoch-validation-"))
model_dir = validation_root / "tiny-gpt2"
model_dir.mkdir()
model_config = GPT2Config(
    vocab_size=64, n_positions=16, n_ctx=16, n_embd=32, n_layer=1, n_head=2,
    bos_token_id=1, eos_token_id=2, pad_token_id=0, use_cache=False,
)
GPT2LMHeadModel(model_config).save_pretrained(model_dir)

def documents(count):
    return [[3 + ((row + offset) % 60) for offset in range(16)] for row in range(count)]

train_dir = validation_root / "train-data"
eval_dir = validation_root / "eval-data"
write_packed_binary_shard(documents(16), train_dir, shard_id="train", vocab_size=64)
write_packed_binary_shard(documents(4), eval_dir, shard_id="eval", vocab_size=64)
count_check = PackedBinDataset.from_directory(
    train_dir, sequence_length=16, split="train", seed=42
)
assert len(count_check) == 16
count_check.close()

class EventRecorder(TrainerCallback):
    def __init__(self):
        self.evaluations = 0
        self.saves = 0

    def on_evaluate(self, state, control):
        del state, control
        self.evaluations += 1

    def on_save_checkpoint(self, state, control):
        del state, control
        self.saves += 1

common = dict(
    accelerator=BACKEND,
    sequence_length=16,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=3,
    learning_rate=1e-3,
    warmup_steps=0,
    logging_steps=1,
)

def run_case(name, *, eval_strategy, save_strategy, epochs, resume_from=None):
    train_rows = PackedBinDataset.from_directory(
        train_dir, sequence_length=16, split="train", seed=42
    )
    eval_rows = (
        PackedBinDataset.from_directory(
            eval_dir, sequence_length=16, split="validation"
        )
        if eval_strategy == "epoch"
        else None
    )
    output_dir = validation_root / name
    events = EventRecorder()
    try:
        trainer = TrainLMTrainer.from_pretrained(
            model_dir,
            train_dataset=train_rows,
            eval_dataset=eval_rows,
            callbacks=[events],
            args=TrainLMTrainingArguments(
                output_dir=output_dir,
                max_steps=None,
                num_train_epochs=epochs,
                eval_strategy=eval_strategy,
                save_strategy=save_strategy,
                max_eval_batches=2 if eval_rows is not None else None,
                **common,
            ),
        )
        result = trainer.train(resume_from_checkpoint=resume_from)
        if BACKEND == "tpu":
            worker = result["worker_summary"]
            log = (output_dir / "train.log").read_text(encoding="utf-8")
            evaluations = log.count('"stage": "evaluation_completed"')
            checkpoints = [Path(path).name for path in worker["committed_checkpoints"]]
            step, micro_step = worker["steps"], worker["micro_steps"]
            world_size, device = worker["world_size"], worker["runtime"]["device"]
            status = result["status"]
        else:
            worker = result
            step, micro_step = result.step, result.micro_step
            world_size = trainer.runtime.world_size
            device = str(trainer.runtime.device)
            status = "completed" if result.phase.value == "finalized" else result.phase.value
            evaluations = events.evaluations
            checkpoints = sorted(
                path.name for path in output_dir.glob("checkpoint-*")
                if (path / "trainer_state.pt").is_file()
            )
        assert status == "completed"
        assert world_size == 1
        assert device.startswith("xla") if BACKEND == "tpu" else device.startswith("tpu")
        expected_evaluations = (2 if name == "fresh" else 1) if eval_strategy == "epoch" else 0
        expected_checkpoints = (2 if name == "fresh" else 1) if save_strategy == "epoch" else 0
        assert evaluations == expected_evaluations, (name, evaluations)
        assert len(checkpoints) == expected_checkpoints, (name, checkpoints)
        return {
            "status": status,
            "steps": step,
            "micro_steps": micro_step,
            "world_size": world_size,
            "device": device,
            "epoch_evaluations": evaluations,
            "committed_checkpoints": checkpoints,
        }
    finally:
        train_rows.close()
        if eval_rows is not None:
            eval_rows.close()

fresh = run_case(
    "fresh", eval_strategy="epoch", save_strategy="epoch", epochs=2
)
assert fresh["steps"] == 6 and fresh["micro_steps"] == 16
first_epoch_checkpoint = validation_root / "fresh" / "checkpoint-3"
marker = "manifest.json" if BACKEND == "tpu" else "trainer_state.pt"
assert (first_epoch_checkpoint / marker).is_file()
resumed = run_case(
    "resumed", eval_strategy="epoch", save_strategy="epoch", epochs=2,
    resume_from=first_epoch_checkpoint,
)
assert resumed["steps"] == 6 and resumed["micro_steps"] == 16

evaluation_only = run_case(
    "evaluation-only", eval_strategy="epoch", save_strategy="no", epochs=1
)
assert evaluation_only["steps"] == 3 and evaluation_only["micro_steps"] == 8
checkpoint_only = run_case(
    "checkpoint-only", eval_strategy="no", save_strategy="epoch", epochs=1
)
assert checkpoint_only["steps"] == 3 and checkpoint_only["micro_steps"] == 8

runtime_packages = {"torch": torch.__version__}
if BACKEND == "native_tpu":
    for package in ("torch-tpu", "libtpu"):
        try:
            runtime_packages[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            runtime_packages[package] = None
else:
    runtime_packages["torch-xla"] = importlib.metadata.version("torch-xla")
report = {
    "schema_version": 1,
    "scope": "epoch_eval_save_single_device_v5e1",
    "source_revision": source_revision,
    "backend": "native_torchtpu" if BACKEND == "native_tpu" else "legacy_pytorch_xla",
    "legacy_xla_path_hardware_validated": BACKEND == "tpu",
    "hardware": {
        "vm_machine_type_entered_by_owner": TPU_VM_MACHINE_TYPE,
        "hardware_identity_verified_by_notebook": False,
    },
    "runtime_packages": runtime_packages,
    "cpu_contract_tests": {
        "status": "passed",
        "exit_code": cpu_tests.returncode,
        "summary": cpu_tests.stdout.strip().splitlines()[-1],
    },
    "fresh_run": fresh,
    "resume_run": {**resumed, "resumed_from": first_epoch_checkpoint.name},
    "evaluation_only": evaluation_only,
    "checkpoint_only": checkpoint_only,
    "performance_certified": False,
}
report_path = validation_root / "tpu_epoch_validation_report.json"
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))
print(f"evidence_report={report_path}")


## 4. Download the report

The report records source revision, package versions, schedule outcomes, and checkpoint names. It labels native TorchTPU and legacy PyTorch/XLA separately. The VM type is owner-entered metadata; verify it in Cloud TPU details before sharing. Training logs and temporary files remain in the runtime.


In [ ]:
try:
    from google.colab import files
except ImportError:
    print(f"Download this report from: {report_path}")
else:
    files.download(str(report_path))
